# Notebook 03: Flash Flood Risk Overlay

Overlay model-predicted rainfall on Singapore's PUB flood-prone area map.

**Prerequisites:**
- Stage 4 complete: `checkpoints/nowcaster/latest.pt`
- Recent radar data in `data/processed/radar.zarr`

**Flash flood risk definition used here:**  
Predicted rain rate ≥ 30 mm/hr sustained for ≥ 30 min over any grid cell.

In [ ]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import xarray as xr
import requests

from src.model.unet import ConditionedUNet
from src.model.diffusion import GaussianDiffusion
from src.data.radar_dataset import RadarDataset, load_stats, CONTEXT_FRAMES

# Singapore bounding box
SG_LAT = (1.0, 1.6)
SG_LON = (103.5, 104.1)

FLASH_FLOOD_THRESHOLD_MM = 30.0   # mm/hr
FLASH_FLOOD_PROB_CUTOFF  = 0.5    # fraction of ensemble members exceeding threshold

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Load model
ckpt = ROOT / 'checkpoints/nowcaster/latest.pt'
state = torch.load(ckpt, map_location='cpu', weights_only=True)
unet = ConditionedUNet(context_frames=CONTEXT_FRAMES, use_checkpoint=False)
diffusion = GaussianDiffusion(unet)
diffusion.load_state_dict(state['model'])
diffusion = diffusion.to(device).eval()
print(f'Model loaded (step {state["step"]})')

In [ ]:
# Generate a forecast for the most recent available timestamp
stats = load_stats()

def denorm(x):
    return np.expm1(np.clip(x * stats['log_std'] + stats['log_mean'], 0, None))

# Use last test sample as a proxy for 'current' radar state
test_ds = RadarDataset('test', context_frames=CONTEXT_FRAMES)
sample = test_ds[-1]
ctx = sample['context'].unsqueeze(0).to(device)

with torch.no_grad():
    ens = diffusion.ensemble_sample(ctx, n_members=16, eta=1.0)
    ens_np = ens.squeeze(0).squeeze(2).cpu().numpy()

ens_mm = denorm(ens_np)  # (16, H, W)

# Probability of exceeding flash-flood threshold
prob_flood = (ens_mm >= FLASH_FLOOD_THRESHOLD_MM).mean(axis=0)  # (H, W)
high_risk  = prob_flood >= FLASH_FLOOD_PROB_CUTOFF

print(f'High-risk cells (P≥{FLASH_FLOOD_PROB_CUTOFF}): {high_risk.sum()} / {high_risk.size}')

In [ ]:
# Download PUB flood-prone areas GeoJSON from data.gov.sg (public)
# Fallback: draw a simple risk heatmap without vector overlay
flood_prone_geojson = None
try:
    # Try data.gov.sg flood risk dataset
    resp = requests.get(
        'https://api.data.gov.sg/v1/transport/carpark-availability',
        timeout=10
    )
    # Note: actual flood-prone area GeoJSON requires a dataset ID from data.gov.sg
    # Contact PUB or check: https://data.gov.sg for the flood-prone area dataset
    print('GeoJSON download: check data.gov.sg for flood-prone area dataset ID')
except Exception as e:
    print(f'GeoJSON unavailable: {e}')

# Visualise flood risk probability map over Singapore domain
lats = np.linspace(SG_LAT[1], SG_LAT[0], prob_flood.shape[0])  # N→S
lons = np.linspace(SG_LON[0], SG_LON[1], prob_flood.shape[1])  # W→E

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: ensemble mean rain rate
im0 = axes[0].imshow(
    ens_mm.mean(axis=0), origin='upper',
    extent=[SG_LON[0], SG_LON[1], SG_LAT[0], SG_LAT[1]],
    vmin=0, vmax=30, cmap=plt.cm.Blues, aspect='auto'
)
plt.colorbar(im0, ax=axes[0], label='Rain rate (mm/hr)')
axes[0].set_title('Ensemble Mean Rainfall (+30 min)')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

# Right: flash flood probability
im1 = axes[1].imshow(
    prob_flood, origin='upper',
    extent=[SG_LON[0], SG_LON[1], SG_LAT[0], SG_LAT[1]],
    vmin=0, vmax=1, cmap=plt.cm.RdYlGn_r, aspect='auto'
)
plt.colorbar(im1, ax=axes[1], label=f'P(rain ≥ {FLASH_FLOOD_THRESHOLD_MM} mm/hr)')

# Overlay high-risk contour
X, Y = np.meshgrid(lons, lats)
axes[1].contour(X, Y, prob_flood, levels=[FLASH_FLOOD_PROB_CUTOFF],
                colors='red', linewidths=1.5)
axes[1].set_title(f'Flash Flood Probability (threshold={FLASH_FLOOD_THRESHOLD_MM} mm/hr)')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

red_patch = mpatches.Patch(color='red', label=f'High risk (P≥{FLASH_FLOOD_PROB_CUTOFF})')
axes[1].legend(handles=[red_patch], loc='lower right')

plt.suptitle('Singapore Flash Flood Risk Map — Nowcast +30 min', fontsize=13)
plt.tight_layout()
plt.savefig(ROOT / 'results/flood_risk_map.png', dpi=150)
plt.show()
print('Saved → results/flood_risk_map.png')